# Objective
Select the disease-specific PRS used in `02_primary_incident_ckm.ipynb`.

For each endpoint, every candidate PGS (from the KPRB PRS request, matched by target outcome) is scored by its C-index boost over a base Cox model:
* Base: `Surv(time, status) ~ age_at_sample + sex`
* PRS: `Surv(time, status) ~ sex + PRS * age_at_sample`, with PRS as continuous or binarized at the top 50/20/10/5%

PRS are scaled **after** excluding outcome-specific prevalent cases, since prevalent cases have higher-than-average PRS and would otherwise shift the scaled distribution.

In [ ]:
suppressPackageStartupMessages({
    library(tidyverse)
    library(survival)
    library(here)
})

# Load Data and Helpers

In [ ]:
# load tte helpers
source(here('helpers', 'prep_incident_time_to_event_df.r'))

In [ ]:
# load metadata
meta <- read.csv(here('data', 'meta_clin.csv'), row.names = 1)
dim(meta)

## Load PRS

In [ ]:
# load prs and preprocess
prs <- read.csv(here('data', 'prs.csv')) %>%
    # remove '_hmPOS_GRCh38' from PGS column
    mutate(PGS = stringr::str_remove(PGS, '_hmPOS_GRCh38')) %>%
    # drop exact duplicate rows (obtain 1 PGS-specific score per patient)
    distinct() %>%
    select(StudyID, PGS, SUM)

stopifnot(!anyDuplicated(prs[c('StudyID', 'PGS')]))

head(prs); dim(prs)

In [ ]:
# load requested PGS IDs and their target outcome
pgs_catalog <- read.csv(here('data/prs_catalog.csv')) %>%
    # PGP (publication) IDs were not delivered as scores
    filter(!stringr::str_starts(PGS.ID, 'PGP')) %>%
    # some PGS IDs are listed under 2 outcomes; keep the first listing
    distinct(PGS.ID, .keep_all = TRUE)

stopifnot(all(pgs_catalog$PGS.ID %in% prs$PGS))

pgs_catalog %>% dplyr::count(Outcome)

# Configuration

In [ ]:
# top-X% cutoffs used to binarize each PRS (name = X, value = quantile)
prs_cutoffs <- c('50' = 0.50, '20' = 0.80, '10' = 0.90, '5' = 0.95)

# all cardiovascular endpoints share the same candidate PGS pool
cvd_pgs_outcomes <- c('CAD', 'CVD', 'MACE_Cardiometabolic', 'MI', 'PAD', 'Stroke')

# outcome = prefix of '<outcome>_status' / '<outcome>_time' in meta_clin.csv
# list order sets the row order of the exported table
endpoints <- list(
  list(label = 'T2D',           outcome = 't2d',         pgs_outcomes = 'T2D'),
  list(label = 'CKD',           outcome = 'ckd',         pgs_outcomes = 'CKD'),
  list(label = 'MASLD',         outcome = 'nafld',       pgs_outcomes = 'Liver Disease'),
  list(label = 'Heart Failure', outcome = 'hf_cm',       pgs_outcomes = cvd_pgs_outcomes),
  list(label = 'Acute MI',      outcome = 'acute_mi',    pgs_outcomes = cvd_pgs_outcomes),
  list(label = 'TIA/Stroke',    outcome = 'tia_stroke',  pgs_outcomes = cvd_pgs_outcomes),
  list(label = 'CHD',           outcome = 'chd_minerva', pgs_outcomes = cvd_pgs_outcomes),
  list(label = 'Total CVD',     outcome = 'total_cvd',   pgs_outcomes = cvd_pgs_outcomes),
  list(label = 'Afib',          outcome = 'afib',        pgs_outcomes = cvd_pgs_outcomes)
)

In [ ]:
# scale each PRS within PGS (long format)
scale_sum_by_pgs <- function(df) {
  df %>%
    group_by(PGS) %>%
    mutate(SUM = scale(SUM, center = TRUE, scale = TRUE)[, 1]) %>%
    ungroup()
}

# delta C-index of continuous and top-X% binarized PRS (PRS * age) over base (age + sex)
assess_prs_cutoff_boost <- function(meta_df, pgs_ids, cutoffs = prs_cutoffs) {
  map_dfr(pgs_ids, function(prs_id) {
    meta_sub <- meta_df %>%
      filter(PGS == prs_id) %>%
      drop_na(age_at_sample, sex, SUM, time, status)

    # binarize SUM at each cutoff within this PGS
    for (nm in names(cutoffs)) {
      meta_sub[[paste0('prs_', nm, 'pct')]] <- as.integer(
        meta_sub$SUM >= quantile(meta_sub$SUM, cutoffs[[nm]])
      )
    }

    fit_c <- function(rhs) {
      summary(coxph(as.formula(paste('Surv(time, status) ~', rhs)), data = meta_sub))$concordance[1]
    }

    base_c <- fit_c('age_at_sample + sex')
    row <- tibble(PGS = prs_id, delta_C = fit_c('sex + SUM*age_at_sample') - base_c)
    for (nm in names(cutoffs)) {
      row[[paste0('delta_C_', nm)]] <- fit_c(paste0('sex + prs_', nm, 'pct*age_at_sample')) - base_c
    }
    row
  })
}

# prepare incident tte data for one endpoint, join candidate PRS, and score them
run_prs_selection <- function(endpoint, meta, prs, pgs_catalog) {
  status_col <- paste0(endpoint$outcome, '_status')
  time_col   <- paste0(endpoint$outcome, '_time')

  pgs_ids <- pgs_catalog %>% filter(Outcome %in% endpoint$pgs_outcomes) %>% pull(PGS.ID)

  meta_df <- prep_incident_time_to_event_df(
      meta, status_col = status_col, time_col = time_col, time_filter = paste(time_col, '> 0')
    ) %>%
    select(StudyID, age_at_sample, sex, time, status) %>%
    inner_join(filter(prs, PGS %in% pgs_ids), by = 'StudyID', relationship = 'one-to-many') %>%
    # scale after excluding prevalent cases
    scale_sum_by_pgs()

  n_pat <- n_distinct(meta_df$StudyID)
  n_inc <- meta_df %>% distinct(StudyID, status) %>% pull(status) %>% sum()
  cat(sprintf('%-14s n = %5d, incident = %4d, candidate PGS = %d\n',
              endpoint$label, n_pat, n_inc, length(pgs_ids)))

  assess_prs_cutoff_boost(meta_df, pgs_ids) %>% mutate(endpoint = endpoint$label)
}

# PRS Selection

In [ ]:
prs_sel <- map_dfr(endpoints, run_prs_selection, meta = meta, prs = prs, pgs_catalog = pgs_catalog)
dim(prs_sel)

In [ ]:
# selected PRS per endpoint (largest continuous delta C); these are the prs_id values in 02_primary_incident_ckm.ipynb
prs_sel %>%
  group_by(endpoint) %>%
  slice_max(delta_C, n = 1) %>%
  ungroup()

# Export

In [ ]:
# arrange by endpoint and delta C, rename, and round
prs_sel_table <- prs_sel %>%
  mutate(endpoint = factor(endpoint, levels = map_chr(endpoints, 'label'))) %>%
  arrange(endpoint, desc(delta_C)) %>%
  transmute(
    `PGS ID`        = PGS,
    `Delta C`       = delta_C,
    `Delta C (50%)` = delta_C_50,
    `Delta C (20%)` = delta_C_20,
    `Delta C (10%)` = delta_C_10,
    `Delta C (5%)`  = delta_C_5,
    Endpoint        = endpoint
  ) %>%
  mutate(across(starts_with('Delta C'), ~ round(.x, 6)))

head(prs_sel_table); dim(prs_sel_table)

# save
write.csv(prs_sel_table, here('outputs/tables', 'PRS_SEL.csv'), row.names = FALSE)